# Figure 3 spatial DES benchmark

This notebook validates and displays the checksum-bound Cesaro et al. (2025) Figure 3 reanalysis. Rankings remain isolated by dataset, scenario, analysis unit, spatial truth, self-pair policy, tie policy, and DES semantics.

In [ ]:
from hashlib import sha256
from json import loads
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd


started = perf_counter()
repo_root = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / 'pyproject.toml').is_file()
)
bundle = (
    repo_root
    / 'benchmarks/results/spatial_des_figure3_reanalysis_20260721'
)
manifest = loads((bundle / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'complete'
for filename, record in manifest['outputs'].items():
    payload = (bundle / filename).read_bytes()
    assert len(payload) == record['bytes']
    assert sha256(payload).hexdigest() == record['sha256']
leaderboard = pd.read_csv(bundle / 'spatial_des_leaderboard.tsv', sep='\t')
print(f'Loaded {len(leaderboard)} method-panel rows in {perf_counter() - started:.3f} s')

## Validate panel-local ranks

Only methods with all eight condition-by-fraction DES strata are rank eligible.

In [ ]:
started = perf_counter()
eligible = leaderboard.loc[leaderboard['rank_eligible']].copy()
eligible['recomputed_rank'] = (
    eligible.groupby('comparison_panel_id', observed=True)['des_median']
    .rank(method='min', ascending=False)
    .astype(int)
)
assert eligible['recomputed_rank'].equals(eligible['median_rank'].astype(int))
columns = [
    'dataset', 'scenario', 'analysis_unit', 'method_display',
    'median_rank', 'des_median', 'des_strata_observed',
    'rank_eligible_fraction_min', 'expected_set_coverage_min',
]
print(f'Validated ranks in {perf_counter() - started:.3f} s')
leaderboard.loc[:, columns]

## Plot median DES

Each axis is one checksum-compatible comparison panel. Incomplete methods remain visible but are marked as not ranked in the table above.

In [ ]:
started = perf_counter()
colors = {
    'scSeqCommDiff': '#D55E00',
    'scDiffCom': '#009E73',
    'CellChat': '#E69F00',
    'LIANA+': '#CC79A7',
}
panels = list(
    leaderboard.groupby('comparison_panel_id', sort=True, observed=True)
)
figure, axes = plt.subplots(
    len(panels), 1, figsize=(9, 3.2 * len(panels)), squeeze=False
)
for axis, (_, panel) in zip(axes.flat, panels, strict=True):
    panel = panel.sort_values('des_median', ascending=True)
    axis.barh(
        panel['method_display'],
        panel['des_median'],
        color=[colors.get(method, '#777777') for method in panel['method_display']],
    )
    first = panel.iloc[0]
    axis.set_title(
        f"{first['dataset']} | {first['scenario']} | {first['analysis_unit']}"
    )
    axis.set_xlabel('Median spatial DES (higher is better)')
    axis.grid(axis='x', color='#D9D9D9', linewidth=0.7)
    axis.set_axisbelow(True)
figure.tight_layout()
print(f'Rendered panels in {perf_counter() - started:.3f} s')